In [1]:
import torch
import time
import datetime
import random
import numpy as np
import polars as pl
from scipy.sparse import csr_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split
import seaborn as sns

In [ ]:
EPOCHS = 10
BATCH_SIZE = 100

In [3]:
torch.set_default_dtype(torch.float32)
if torch.cuda.is_available():
    torch.set_default_device(0)
    print("Running on the GPU")
else:
    print("Running on the CPU")

Running on the CPU


[bood_id_map.csv](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/book_id_map.csv)
[user_id_map.csv](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/user_id_map.csv)
[goodreads_books.json.gz](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_books.json.gz)
[goodreads_reviews_dedup.json.gz](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_reviews_dedup.json.gz)
[goodreads_interactions.csv](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions.csv)
[goodreads_interactions_dedup.json.gz](https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions_dedup.json.gz)

In [4]:
# book_ids = pl.read_csv('./data/book_id_map.csv')
# user_ids = pl.read_csv('./data/user_id_map.csv')
# books = pl.read_ndjson('./data/goodreads_books.json')
# reviews = pl.read_ndjson('./data/goodreads_reviews_dedup.json')
# interactions = pl.read_csv('./data/goodreads_interactions.csv')

# book_ids.write_ipc('./data/book_id_map.feather')
# user_ids.write_ipc('./data/user_id_map.feather')
# books.write_ipc('./data/goodreads_books.feather')
# reviews.write_ipc('./data/goodreads_reviews_dedup.feather')
# interactions.write_ipc('./data/goodreads_interactions.feather')

In [5]:
book_ids = pl.read_ipc('./data/book_id_map.feather')
user_ids = pl.read_ipc('./data/user_id_map.feather')
books = pl.read_ipc('./data/goodreads_books.feather')
reviews = pl.read_ipc('./data/goodreads_reviews_dedup.feather')
interactions = pl.read_ipc('./data/goodreads_interactions.feather')

In [6]:
book_ids.head()

book_id_csv,book_id
i64,i64
0,34684622
1,34536488
2,34017076
3,71730
4,30422361


In [7]:
user_ids.head()

user_id_csv,user_id
i64,str
0,"""8842281e1d1347389f2ab93d60773d…"
1,"""72fb0d0087d28c832f15776b0d9365…"
2,"""ab2923b738ea3082f5f3efcbbfacb2…"
3,"""d986f354a045ffb91234e4af4d1b12…"
4,"""7504b2aee1ecb5b2872d3da381c6c9…"


In [8]:
books.head()

isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
str,str,list[str],str,str,list[struct[2]],str,str,str,str,list[str],str,str,str,list[struct[2]],str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""0312853122""","""1""",[],"""US""","""""","[{""3"",""to-read""}, {""1"",""p""}, … {""1"",""biography""}]","""""","""false""","""4.00""","""""",[],"""""","""Paperback""","""https://www.goodreads.com/book…","[{""604031"",""""}]","""St. Martin's Press""","""256""","""1""","""9780312853129""","""9""","""""","""1984""","""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…","""5333265""","""3""","""5400751""","""W.C. Fields: A Life on Film""","""W.C. Fields: A Life on Film"""
"""0743509986""","""6""",[],"""US""","""""","[{""2634"",""to-read""}, {""160"",""fiction""}, … {""2"",""general""}]","""""","""false""","""3.23""","""B000FC0PBC""","[""8709549"", ""17074050"", … ""307575""]","""Anita Diamant's international …","""Audio CD""","""https://www.goodreads.com/book…","[{""626222"",""""}]","""Simon & Schuster Audio""","""""","""1""","""9780743509985""","""10""","""Abridged""","""2001""","""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…","""1333909""","""10""","""1323437""","""Good Harbor""","""Good Harbor"""
"""""","""7""","[""189911""]","""US""","""eng""","[{""58"",""to-read""}, {""15"",""fantasy""}, … {""1"",""read-in-my-20s""}]","""B00071IKUY""","""false""","""4.03""","""""","[""19997"", ""828466"", … ""1597281""]","""Omnibus book club edition cont…","""Hardcover""","""https://www.goodreads.com/book…","[{""10333"",""""}]","""Nelson Doubleday, Inc.""","""600""","""""","""""","""""","""Book Club Edition""","""1987""","""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…","""7327624""","""140""","""8948723""","""The Unschooled Wizard (Sun Wol…","""The Unschooled Wizard (Sun Wol…"
"""0743294297""","""3282""",[],"""US""","""eng""","[{""7615"",""to-read""}, {""728"",""chick-lit""}, … {""6"",""bookworm-bitches""}]","""""","""false""","""3.49""","""B002ENBLOK""","[""6604176"", ""6054190"", … ""3134684""]","""Addie Downs and Valerie Adler …","""Hardcover""","""https://www.goodreads.com/book…","[{""9212"",""""}]","""Atria Books""","""368""","""14""","""9780743294294""","""7""","""""","""2009""","""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…","""6066819""","""51184""","""6243154""","""Best Friends Forever""","""Best Friends Forever"""
"""0850308712""","""5""",[],"""US""","""""","[{""32"",""to-read""}, {""3"",""runes""}, … {""1"",""magick-and-myth""}]","""""","""false""","""3.40""","""""",[],"""""","""""","""https://www.goodreads.com/book…","[{""149918"",""""}]","""""","""""","""""","""9780850308716""","""""","""""","""""","""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…","""287140""","""15""","""278577""","""Runic Astrology: Starcraft and…","""Runic Astrology: Starcraft and…"


In [9]:
reviews.head()

user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments
str,str,str,i64,str,str,str,str,str,i64,i64
"""8842281e1d1347389f2ab93d60773d…","""24375664""","""5cd416f3efc3f944fce4ce2db2290d…",5,"""Mind blowingly cool. Best scie…","""Fri Aug 25 13:55:02 -0700 2017""","""Mon Oct 09 08:55:59 -0700 2017""","""Sat Oct 07 00:00:00 -0700 2017""","""Sat Aug 26 00:00:00 -0700 2017""",16,0
"""8842281e1d1347389f2ab93d60773d…","""18245960""","""dfdbb7b0eb5a7e4c26d59a937e2e5f…",5,"""This is a special book. It sta…","""Sun Jul 30 07:44:10 -0700 2017""","""Wed Aug 30 00:00:26 -0700 2017""","""Sat Aug 26 12:05:52 -0700 2017""","""Tue Aug 15 13:23:18 -0700 2017""",28,1
"""8842281e1d1347389f2ab93d60773d…","""6392944""","""5e212a62bced17b4dbe41150e5bb90…",3,"""I haven't read a fun mystery b…","""Mon Jul 24 02:48:17 -0700 2017""","""Sun Jul 30 09:28:03 -0700 2017""","""Tue Jul 25 00:00:00 -0700 2017""","""Mon Jul 24 00:00:00 -0700 2017""",6,0
"""8842281e1d1347389f2ab93d60773d…","""22078596""","""fdd13cad0695656be99828cd75d6eb…",4,"""Fun, fast paced, and disturbin…","""Mon Jul 24 02:33:09 -0700 2017""","""Sun Jul 30 10:23:54 -0700 2017""","""Sun Jul 30 15:42:05 -0700 2017""","""Tue Jul 25 00:00:00 -0700 2017""",22,4
"""8842281e1d1347389f2ab93d60773d…","""6644782""","""bd0df91c9d918c0e433b9ab3a9a5c4…",4,"""A fun book that gives you a se…","""Mon Jul 24 02:28:14 -0700 2017""","""Thu Aug 24 00:07:20 -0700 2017""","""Sat Aug 05 00:00:00 -0700 2017""","""Sun Jul 30 00:00:00 -0700 2017""",8,0


In [10]:
interactions.head()

user_id,book_id,is_read,rating,is_reviewed
i64,i64,i64,i64,i64
0,948,1,5,0
0,947,1,5,1
0,946,1,5,0
0,945,1,5,0
0,944,1,5,0


In [11]:
interactions = interactions.filter(pl.col('is_read').eq(1))
interactions = interactions.drop('is_read')
interactions = interactions.filter(pl.col('user_id').is_duplicated())

In [12]:
def normalize_ids(df: pl.DataFrame, id: str):
    uniq_ids = set(df[id].unique())
    ids_to_indices = { orig_id : new_id for new_id, orig_id in enumerate(uniq_ids) }
    new_ids = df.with_columns(pl.col(id).replace(ids_to_indices).cast(pl.Int64))
    indices_to_ids = { new_id : orig_id for orig_id, new_id in ids_to_indices.items()}
    return new_ids, indices_to_ids

In [13]:
interactions, _ = normalize_ids(interactions, 'book_id')
interactions, _ = normalize_ids(interactions, 'user_id')

interactions_n_books = interactions['book_id'].n_unique()

In [14]:
gss = GroupShuffleSplit(n_splits=5)

interactions_train_idx, interactions_test_idx = next(gss.split(X=interactions, groups=interactions['user_id']))
interactions_train: pl.Dataframe = interactions[interactions_train_idx]
interactions_test:  pl.Dataframe = interactions[interactions_test_idx]

interactions_train, interactions_train_map_id = normalize_ids(interactions_train, 'user_id')
interactions_test,  interactions_test_map_id  = normalize_ids(interactions_test, 'user_id')

interactions_train_csr = csr_matrix((interactions_train['rating'], (interactions_train['user_id'], interactions_train['book_id'])), shape=(interactions_train['user_id'].n_unique(), interactions_n_books))

In [15]:
interactions_test_seen,   interactions_test_unseen         = train_test_split(interactions_test, train_size=0.7, stratify=interactions_test['user_id'])

interactions_test_seen,   interactions_test_seen_map_id    = normalize_ids(interactions_test_seen,   'user_id')
interactions_test_unseen, interactions_test_unseen_map_id  = normalize_ids(interactions_test_unseen, 'user_id')

interactions_test_seen_csr = csr_matrix((interactions_test_seen['rating'], (interactions_test_seen['user_id'], interactions_test_seen['book_id'])), shape=(interactions_test_seen['user_id'].n_unique(), interactions_n_books))

In [16]:
train_crow_idx = interactions_train_csr.tocoo().coords[0]
np.append(train_crow_idx, interactions_train_csr.getnnz(axis=0))
train_col_idx = interactions_train_csr.tocoo().coords[1]
np.append(train_crow_idx, interactions_train_csr.getnnz(axis=1))
train_values = interactions_train_csr.tocoo().data

test_crow_idx = interactions_test_seen_csr.tocoo().coords[0]
test_col_idx = interactions_test_seen_csr.tocoo().coords[1]
test_values = interactions_test_seen_csr.tocoo().data

interactions_features = torch.sparse_csr_tensor(crow_indices=torch.tensor(train_crow_idx, dtype=torch.int64),
                                                col_indices=torch.tensor(train_col_idx, dtype=torch.int64),
                                                values=torch.tensor(train_values, dtype=torch.int64))
interactions_labels = torch.sparse_csr_tensor(crow_indices=torch.tensor(test_crow_idx, dtype=torch.int64),
                                              col_indices=torch.tensor(test_col_idx, dtype=torch.int64),
                                              values=torch.tensor(test_values, dtype=torch.int64))

/var/folders/zj/k5n_79qd4r982mm8sxt_dhb00000gn/T/ipykernel_90774/1049076643.py:11: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/Context.cpp:767.)
  interactions_features = torch.sparse_csr_tensor(crow_indices=torch.tensor(train_crow_idx, dtype=torch.int64),
/var/folders/zj/k5n_79qd4r982mm8sxt_dhb00000gn/T/ipykernel_90774/1049076643.py:11: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorI

In [17]:
class Timer(object):
    def __init__(self, name=None, filename=None):
        self.name = name
        self.filename = filename

    def __enter__(self):
        self.tstart = time.time()

    def __exit__(self, type, value, traceback):
        message = 'Elapsed: %.2f seconds' % (time.time() - self.tstart)
        if self.name:
            message = '[%s] ' % self.name + message
        print(message)
        if self.filename:
            with open(self.filename,'a') as file:
                print(str(datetime.datetime.now())+": ",message,file=file)

In [18]:
print(interactions['user_id'].n_unique())
print(interactions_n_books)
print(interactions_train_csr.nnz)

807178
2339175
89700309


In [19]:
class AE(torch.nn.Module):
    def __init__(self):
        super(AE, self).__init__()
        self.model = torch.nn.Sequential()
        # 31189 is the highest primefactor of the number of books available on good reads
        self.model.append(torch.nn.Linear(interactions_n_books, interactions_n_books // 31189))
        self.model.append(torch.nn.ReLU())
        self.model.append(torch.nn.Linear(interactions_n_books // 31189, interactions_n_books // 31189))
        self.model.append(torch.nn.ReLU())
        self.model.append(torch.nn.Linear(interactions_n_books // 31189, interactions_n_books))
        self.model.append(torch.nn.Sigmoid())
        
    def forward(self, x):
        return self.model(x)

In [20]:
#loader = torch.utils.data.DataLoader(dataset=interactions_train_csr.data, batch_size=BATCH_SIZE, shuffle=True)
auto_encoder = AE()
loss_function = torch.nn.MSELoss()
optimizer = torch.optim.Adam(auto_encoder.parameters(), lr=0.001, weight_decay=0.00000001)

In [21]:
training_loss = []

In [ ]:
with Timer('Total time'):
    with Timer('Training time'):
        for epoch in range(EPOCHS):
            dataset = set(interactions_train_csr.tocoo().coords[0].tolist())
            for x_idx in range(0, len(dataset), BATCH_SIZE):
                x_arr = np.array(interactions_train_csr.getrow(list(dataset)[x_idx]).toarray())
                for x_idx_idx in range(x_idx+1, x_idx+BATCH_SIZE):
                    if x_idx_idx < len(dataset):
                        np.append(x_arr, interactions_train_csr.getrow(list(dataset)[x_idx_idx]).toarray(), axis=1)
                x = torch.tensor(x_arr, dtype=torch.float32)
                pred = auto_encoder(x)
                loss = loss_function(pred, x)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                training_loss.append(loss.item())

[Training time] Elapsed: 15271.84 seconds
[Total time] Elapsed: 15271.84 seconds


IndexError: list index out of range

In [ ]:
sns.lineplot(training_loss)